In [24]:
import os
import warnings

if 'Modeling' in os.path.abspath("").split('/'):
    os.chdir('..')
if 'Notebooks' in os.path.abspath("").split('/'):
    os.chdir('..')

project_root = os.path.abspath("")

warnings.filterwarnings('ignore')

In [25]:
import numpy as np

In [26]:
loaded = np.load('./Generated/Spectrums/exec_morlets.npz')

In [27]:
results_arr = []
i = 0
while f'power_{i}' in loaded:
    power = loaded[f'power_{i}']
    phase = loaded[f'phase_{i}']
    s_id = int(loaded[f'subject_id_{i}'])
    t_id = int(loaded[f'trial_id_{i}'])
    gender = str(loaded[f'gender_{i}'])
    handiness = str(loaded[f'handiness_{i}'])
    age = int(loaded[f'age_{i}'])
    label = int(loaded[f'label_{i}'])
    img = loaded[f'img_{i}']
    task_type = str(loaded[f'task_type_{i}'])
    
    results_arr.append([power, phase, s_id, t_id, gender, handiness, age, label, img, task_type])
    i += 1

power, phase, s_id, t_id, gender, handiness, age, label, img, task_type = results_arr[0]

In [28]:
import psutil
import os

process = psutil.Process(os.getpid())
print(f"Используется памяти: {process.memory_info().rss / 1024 ** 2:.2f} MB")

Используется памяти: 37989.48 MB


In [29]:
len(results_arr)

1260

In [30]:
subject_id_set  = set()
trial_id_set    = set()
gender_set      = set()
handiness_set   = set()
age_set         = set()
labels_set      = set()
task_type_set   = set()

for power, phase, s_id, t_id, gender, handiness, age, label, img, task_type in results_arr:
    labels_set.add(label)
    task_type_set.add(task_type)
    subject_id_set.add(s_id)
    trial_id_set.add(t_id)
    gender_set.add(gender)
    handiness_set.add(handiness)  # исправлено имя переменной, чтобы не перезаписывать множество
    age_set.add(age)

print("Labels:", labels_set)
print("Task types:", task_type_set)
print("Subject IDs:", subject_id_set)
print("Trial IDs:", trial_id_set)
print("Genders:", gender_set)
print("Handiness:", handiness_set)
print("Ages:", age_set)

Labels: {0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, -1}
Task types: {'r', 'g'}
Subject IDs: {1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34}
Trial IDs: {1, 2}
Genders: {'n', 'm', 'f'}
Handiness: {'r', 'n', 'l'}
Ages: {32, 41, 18, 19, 20, 21, 22, 23, 30}


## Create dir structure

In [32]:
def mkdir_p(path):
    try:
        os.mkdir(path)
    except FileExistsError:
        pass
    except PermissionError:
        print(f"Permission denied: Unable to create '{path}'.")
    except Exception as e:
        print(f"An error occurred: {e}")

In [34]:
last_s_id = None
last_t_id = None
current_trial_path = None
morlets_path = f'{project_root}/Generated/Spectrums/exec_morlets'
mkdir_p(morlets_path)
save_dict = {}
for power, phase, s_id, t_id, gender, handiness, age, label, img, task_type in results_arr:
    if s_id != last_s_id or t_id != last_t_id:
        if last_s_id is not None:
            current_subject_path = f'{morlets_path}/S_{last_s_id}'
            mkdir_p(current_subject_path)
            current_trial_path = f'{current_subject_path}/Trial_{last_t_id}'
            mkdir_p(current_trial_path)
            np.savez(f'{current_trial_path}/exec_morlets_{last_s_id}_{last_t_id}.npz', **save_dict)
        save_dict = {}
        last_s_id = s_id
        last_t_id = t_id

    save_dict[f'power_{i}'] = power       # (ch, freq, time)
    save_dict[f'phase_{i}'] = phase   # (ch, freq, time)
    save_dict[f'subject_id_{i}'] = np.array(s_id)
    save_dict[f'trial_id_{i}'] = np.array(t_id)
    save_dict[f'gender_{i}'] = np.array(gender, dtype='U1')
    save_dict[f'handiness_{i}']  = np.array(handiness, dtype='U1')
    save_dict[f'age_{i}'] = np.array(age, dtype=int)
    save_dict[f'label_{i}']  = np.array(label, dtype=int)
    save_dict[f'img_{i}'] = np.array(img, dtype=int)
    save_dict[f'task_type_{i}'] = np.array(task_type, dtype='U1')

if current_trial_path is not None:
    np.savez(f'{current_trial_path}/exec_morlets_{last_s_id}_{last_t_id}.npz', **save_dict)
